# 02 · Tratamento: `breakhis_raw.csv` → `breakhis.csv`

Aplica as decisões tiradas da `01_eda.ipynb` e
grava o CSV limpo, com as mesmas colunas do bruto.

- Imagem repetida entre benign e malignant: sai dos dois grupos 
- 14-13412: mesmas 123 imagens em DC e LC -> manter só em LC
- LC 14-12204: 2 pares repetidos -> deduplicar (fica a menor `seq`) 
- Pastas com o mesmo número de registro -> paciente = `id_paciente` (ignora a `secao`)
- 14-21998 (F/PT) e 14-29960 (A/F) -> descartar as imagens de F 

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

plt.rcParams["figure.dpi"] = 72
%config InlineBackend.figure_formats = ["jpeg"]

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

# ordem: seen (DC, F, MC, TA), unseen da meta-validação (A, LC), unseen do meta-teste (PT, PC) — D2
SUBTIPOS = ["DC", "F", "MC", "TA", "A", "LC", "PT", "PC"]
PAPEL = dict(DC="seen", F="seen", MC="seen", TA="seen", A="unseen mval", LC="unseen mval", PT="unseen mtst", PC="unseen mtst")
MAGS = [40, 100, 200, 400]

raw = pd.read_csv(DATA / "breakhis_raw.csv", keep_default_na=False)
len(raw)

7909

## Limpeza

In [2]:
dropar = (
    (raw.groupby("rgb_hash").malignidade.transform("nunique") > 1)                    # mesma imagem em benign e malignant
    | ((raw.id_paciente == "14-13412") & (raw.id_subtipo == "DC"))               # 14-13412: mesmas 123 imagens em DC e LC -> manter só em LC
    | (raw.id_paciente.isin(["14-21998", "14-29960"]) & (raw.id_subtipo == "F"))  # 14-21998 (F/PT) e 14-29960 (A/F) -> descartar as imagens de F 
)
clean = raw[~dropar].sort_values("seq").drop_duplicates("rgb_hash").sort_index()  # dedup


clean.to_csv(DATA / "breakhis.csv", index=False)

In [3]:
def resumo(d):
    return d.groupby("id_subtipo").agg(imagens=("path", "size"), pacientes=("id_paciente", "nunique"))

final = pd.concat({"bruta": resumo(raw), "limpa": resumo(clean)}, axis=1).loc[SUBTIPOS]
final.loc["total"] = [len(raw), raw.id_paciente.nunique(), len(clean), clean.id_paciente.nunique()]
final

bruta             limpa          
           imagens pacientes imagens pacientes
id_subtipo                                    
DC            3451        38    3328        37
F             1014         7     688         5
MC             792         7     792         7
TA             569         6     569         6
A              444         2     444         2
LC             626         4     624         4
PT             453         3     453         3
PC             560         6     560         6
total         7909        70    7458        70

In [4]:
pd.crosstab(clean.id_subtipo, clean.magnificacao, margins=True, margins_name="total").loc[SUBTIPOS + ["total"]]

magnificacao,40,100,200,400,total
id_subtipo,,,,,
DC,832,870,864,762,3328
F,167,181,180,160,688
MC,205,222,196,169,792
TA,149,150,140,130,569
A,114,113,111,106,444
LC,154,170,163,137,624
PT,109,121,108,115,453
PC,145,142,135,138,560
total,1875,1969,1897,1717,7458


## Viabilidade dos k-shots no protocolo por paciente (P-PAC)

Simulação do sorteio planejado, sobre o CSV limpo, por
classe e magnificação: embaralhar os pacientes, colocá-los no support até somar **k**
imagens e exigir que os pacientes **restantes** tenham pelo menos **k_tst = 15**
imagens para o query. O valor é a fração de 1.000 sorteios em que isso funciona
(1,00 = sempre viável).

- **Classes unseen** (A, LC, PT, PC): usam todos os pacientes da classe, porque
  aparecem numa etapa só.
- **Classes seen** (DC, F, MC, TA): a divisão entre etapas ainda não existe. Usamos o
  tamanho previsto para mval/mtst, **20% dos pacientes da classe, com mínimo de 2**
  (ex.: DC → 7, TA → 2), sorteados ao acaso a cada repetição.

No P-IMG basta a classe ter k + 15 imagens na etapa, o que sempre ocorre.

In [5]:
K, K_TST = [1, 5, 10, 20], 15


def viavel(contagens, k, n_pac=None, n=1000, seed=0):
    """Fração de sorteios em que support (≥ k imagens) e query (≥ K_TST) cabem em pacientes disjuntos."""
    rng = np.random.default_rng(seed)
    ok = 0
    for _ in range(n):
        c = rng.permutation(contagens)[:n_pac]
        i = np.searchsorted(np.cumsum(c), k)  # primeiro paciente com o support completo
        ok += i < len(c) and c[i + 1:].sum() >= K_TST
    return ok / n


n = clean.groupby(["id_subtipo", "magnificacao", "id_paciente"]).size()
linhas = {}
for s in SUBTIPOS:
    n_total = clean.loc[clean.id_subtipo == s, "id_paciente"].nunique()
    n_pac = max(2, round(0.2 * n_total)) if PAPEL[s] == "seen" else None
    for k in K:
        linhas[(s, k)] = {m: viavel(n.loc[(s, m)].to_numpy(), k, n_pac) for m in MAGS}
viab = pd.DataFrame(linhas).T
viab.index.names = ["id_subtipo", "k"]
viab.style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)